In [16]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [17]:
path = "/Users/phandangvu/Desktop/archive"

courses = pd.read_csv(os.path.join(path, "courses.csv"))
assessments = pd.read_csv(os.path.join(path, "assessments.csv"))
vle = pd.read_csv(os.path.join(path, "vle.csv"))
studentInfo = pd.read_csv(os.path.join(path, "studentInfo.csv"))
studentRegistration = pd.read_csv(os.path.join(path, "studentRegistration.csv"))
student_assessment = pd.read_csv(os.path.join(path, "studentAssessment.csv"))
studentVle = pd.read_csv(os.path.join(path, "studentVle.csv"))

# Hiển thị thử
print("courses")
display(courses.head())

print("assessments")
display(assessments[assessments["code_module"] == "AAA"])

print("vle")
display(vle.head())
display(vle["activity_type"].unique())

print("studentInfo")
display(studentInfo.head())

print("studentRegistration")
display(studentRegistration.head())

print("studentAssessment")
display(studentAssessment.head())   

print("studentVle")
display(studentVle.head())




courses


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240


assessments


,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0
3,AAA,2013J,1755,TMA,166.0,20.0
4,AAA,2013J,1756,TMA,215.0,30.0
5,AAA,2013J,1757,Exam,NaN,100.0
6,AAA,2014J,1758,TMA,19.0,10.0
7,AAA,2014J,1759,TMA,54.0,20.0
8,AAA,2014J,1760,TMA,117.0,20.0
9,AAA,2014J,1761,TMA,166.0,20.0


vle


...

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN


studentAssessment


,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.0
1,1752,28400,22,0,70.0
2,1752,31604,17,0,72.0
3,1752,32885,26,0,69.0
4,1752,38053,19,0,79.0


studentVle


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


In [18]:
# --- BƯỚC 1: XỬ LÝ ĐIỂM SỐ ---
# 1. Tìm các bài kiểm tra có người nộp
submitted_assessments = student_assessment['id_assessment'].unique()
valid_assessments = assessments[assessments['id_assessment'].isin(submitted_assessments)].copy()

# 2. Tính Tổng trọng số thực tế
module_weights = valid_assessments.groupby(['code_module', 'code_presentation'])['weight'].sum().reset_index()
module_weights.rename(columns={'weight': 'valid_total_weight'}, inplace=True)

# 3. Kết nối bảng điểm
df_scores = pd.merge(
    student_assessment, 
    valid_assessments[['id_assessment', 'code_module', 'code_presentation', 'weight']], 
    on='id_assessment', 
    how='inner'
)
df_scores['score'] = df_scores['score'].fillna(0)
df_scores['weighted_score'] = df_scores['score'] * df_scores['weight']

# 4. Gom nhóm tính tổng
student_total_scores = df_scores.groupby(['id_student', 'code_module', 'code_presentation']).agg(
    raw_weighted_sum=('weighted_score', 'sum'),
    mean_score=('score', 'mean')
).reset_index()

student_total_scores = pd.merge(student_total_scores, module_weights, on=['code_module', 'code_presentation'], how='left')

# 5. Hàm chuẩn hóa
def calculate_normalized_score(row):
    if row['valid_total_weight'] > 0:
        return row['raw_weighted_sum'] / row['valid_total_weight']
    else:
        return row['mean_score']

student_total_scores['average_score'] = student_total_scores.apply(calculate_normalized_score, axis=1)
student_total_scores['average_score'] = student_total_scores['average_score'].clip(upper=100)
final_scores = student_total_scores[['id_student', 'code_module', 'code_presentation', 'average_score']]

In [ ]:
# --- BƯỚC 2: XỬ LÝ VLE, CHIA THÀNH 5 NHÓM HOẠT ĐỘNG ---
df_vle_merged = studentVle.merge(vle[['id_site', 'activity_type']], on='id_site', how='left')

def map_5_categories(a_type):
    if a_type in ['resource', 'outcontent', 'url', 'page', 'folder', 'htmlactivity']:
        return "content_resource_sum_click"
    elif a_type in ['quiz', 'externalquiz', 'questionnaire', 'dataplus']:
        return "assessment_and_feedback_sum_click"
    elif a_type in ['glossary', 'ouilluminate', 'repeatactivity']:
        return "tool_sum_click"
    elif a_type in ['homepage', 'subpage', 'sharedsubpage']:
        return "navigation_pages_sum_click"
    else:
        return "interaction_and_discussion_sum_click"

df_vle_merged['vle_category'] = df_vle_merged['activity_type'].apply(map_5_categories)

vle_pivoted = pd.pivot_table(
    df_vle_merged,
    values='sum_click',
    index=['id_student', 'code_module', 'code_presentation'],
    columns='vle_category',
    aggfunc='sum',
    fill_value=0
).reset_index()

In [21]:
# --- BƯỚC 3: KẾT HỢP TẤT CẢ ---
# Lấy các cột từ studentInfo
cols_student_info = [
    'id_student', 'code_module', 'code_presentation', 
    'gender', 'region', 'highest_education', 'imd_band', 
    'age_band', 'num_of_prev_attempts', 'studied_credits', 'final_result'
]
df_master = studentInfo[cols_student_info].copy()

# date_registration từ studentRegistration
df_master = df_master.merge(
    studentRegistration[['id_student', 'code_module', 'code_presentation', 'date_registration']], 
    on=['id_student', 'code_module', 'code_presentation'], 
    how='left'
)

#thêm calculated_score
df_master = df_master.merge(final_scores, on=['id_student', 'code_module', 'code_presentation'], how='left')

#thêm 5 lại hạot động
df_master = df_master.merge(vle_pivoted, on=['id_student', 'code_module', 'code_presentation'], how='left')

# Điền 0 cho các sinh viên không có điểm hoặc không click VLE
df_master['average_score'] = df_master['average_score'].fillna(0)
vle_cols = ["content_resource", "assessment_and_feedback", "tool", "navigation_pages", "interaction_and_discussion"]
df_master[vle_cols] = df_master[vle_cols].fillna(0)

# Sắp xếp lại để final_result ở cuối cùng
cols = [c for c in df_master.columns if c != 'final_result'] + ['final_result']
df_master = df_master[cols]

# Hiển thị kết quả
print("Cấu trúc bảng Master:")
display(df_master.head())

Cấu trúc bảng Master:


,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,date_registration,average_score,assessment_and_feedback,content_resource,interaction_and_discussion,navigation_pages,tool,final_result
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,-159.0,82.4,0.0,18.0,746.0,170.0,0.0,Pass
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,-53.0,65.4,10.0,60.0,954.0,411.0,0.0,Pass
2,30268,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,-92.0,0.0,0.0,8.0,192.0,81.0,0.0,Withdrawn
3,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,-52.0,76.3,2.0,109.0,1470.0,576.0,1.0,Pass
4,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,-176.0,55.0,0.0,59.0,688.0,283.0,4.0,Pass


In [ ]:

df_master.to_csv('final_data.csv', index=False, encoding='utf-8-sig')

